In [1]:
import pandas as pd
import os
from pydub import AudioSegment
from pydub.playback import play

def get_time(row):
    """
    this function returns the start and end time for the interval in milliseconds

    Parameters
    ----------
    row : pandas Series
        a row of the dataframe containing the filename, start_datetime and end_datetime
    Returns
    -------
    Start : int
        the start time of the interval in seconds
    End : int
        the end time of the interval in seconds
    """

    initial_time = row["filename"][11:-4]
    initial_time = int(initial_time[:2])*3600 + int(initial_time[3:5])*60 + int(initial_time[6:8])
    initial_time = initial_time * 1000

    Start = row['start_datetime'][11:26]
    End = row['end_datetime'][11:26]
    Start=int(float(Start[:2]))*3600 + int(float(Start[3:5]))*60 + int(float(Start[6:8])) + float("0." + Start[9:])
    End=int(float(End[:2]))*3600 + int(float(End[3:5]))*60 + int(float(End[6:8])) + float("0." + End[9:])

    Start *= 1000
    End *= 1000

    Start = Start - initial_time
    End = End - initial_time

    return Start, End


os.makedirs('waves', exist_ok=False)

i = 0 # this iterator is used for naming uniqueness, since in our first run and after 2 hours, alot were overwritten(exactly 32,142)
for file in os.scandir('biodcase_development_set/train/annotations'):
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file.path)
        
        for index, row in df.iterrows():
            i+=1
            wave_name=row['filename']
            folder_name = row['dataset']
            wav_file = AudioSegment.from_file(file='biodcase_development_set/train/audio/' +folder_name+ '/' +wave_name, format="wav")
           
            Start, End = get_time(row)
            newAudio=wav_file[Start:End]

            newfilename = row["annotation"] + '_' + str(row["low_frequency"]) + '_' + str(row["high_frequency"]) + "_" + str(i) + ".wav"
            newAudio.export('waves/' + newfilename, format="wav")

In [2]:
csv_lines = 0
for file in os.scandir('biodcase_development_set/train/annotations'):
    if file.is_file() and file.name.endswith('.csv'):
        csv_lines += len(pd.read_csv(file.path))

print("number of lines in train csv files is: ", csv_lines)

number of lines in train csv files is:  58570


In [3]:
wave_events = 0
for file in os.scandir('waves'):
    if file.is_file() and file.name.endswith('.wav'):
        wave_events += 1

print("number of wave events cutted out: ", wave_events)

number of wave events cutted out:  58570


# implement spectrogram

In [ ]:
# will run this in test.py file, since in REPL the same python script uses much more CPU and stacks RAM nonestop increasingly
# since by default the notebook wants to print out the result (the calculated spectrogram)


import csv
from pathlib import Path

import numpy as np
from matplotlib import mlab
from scipy.io import wavfile


WAV_DIR = Path("./waves")
ARRAY_OUTPUT_PATH = Path("./spectrograms_arrays.npy")
INDEX_OUTPUT_PATH = Path("./spectrograms_arrays_index.csv")

NFFT = 512
WINSIZE = 256
OVERLAP_PERCENT = 98
SCALE = "default"


def convert_wav_to_float(data: np.ndarray) -> np.ndarray:
    """Match the notebook conversion so plotting behaves consistently across WAV dtypes."""
    if data.dtype == np.uint8:
        data = (data - 128) / 128.0
    elif data.dtype == np.int16:
        data = data / 32768.0
    elif data.dtype == np.int32:
        data = data / 2147483648.0
    return data.astype(np.float32, copy=False)


def parse_filename(stem: str) -> tuple[str, float, float]:
    """Expected pattern: <label>_<low_freq>_<high_freq>_<index>.wav"""
    parts = stem.split("_")
    if len(parts) != 4:
        raise ValueError(f"Unexpected filename format: {stem}")

    label, low_str, high_str, _ = parts
    return label, float(low_str), float(high_str)


def load_wav_mono(path: Path) -> tuple[int, np.ndarray]:
    sample_rate, data = wavfile.read(path)
    data = convert_wav_to_float(data)

    if data.ndim > 1:
        data = data.mean(axis=1)

    return sample_rate, data


def compute_spectrogram_array(signal: np.ndarray, sample_rate: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Keep spectrogram math aligned with matplotlib.specgram settings:
    nfft=512 | winsize=256 | overlap=98 | scale=default.
    """
    noverlap = int(round(WINSIZE * (OVERLAP_PERCENT / 100.0)))
    noverlap = min(noverlap, WINSIZE - 1)

    power, freqs, times = mlab.specgram(
        x=signal,
        NFFT=WINSIZE,
        Fs=sample_rate,
        noverlap=noverlap,
        pad_to=NFFT,
    )

    if SCALE == "default" or SCALE == "dB":
        # "default" uses dB for PSD spectrograms.
        power = 10.0 * np.log10(np.maximum(power, np.finfo(np.float32).tiny))

    return (
        power.astype(np.float32, copy=False),
        freqs.astype(np.float32, copy=False),
        times.astype(np.float32, copy=False),
    )


def append_event_arrays(
    stream_file,
    csv_writer: csv.writer,
    record_index: int,
    wav_path: Path,
) -> None:
    label, low_freq, high_freq = parse_filename(wav_path.stem)
    sample_rate, signal = load_wav_mono(wav_path)
    spectrogram, freqs, times = compute_spectrogram_array(signal, sample_rate)

    low = min(low_freq, high_freq)
    high = max(low_freq, high_freq)
    band_mask = (freqs >= low) & (freqs <= high)
    spectrogram = spectrogram[band_mask, :]
    freqs = freqs[band_mask]

    spec_offset = stream_file.tell()
    np.save(stream_file, spectrogram, allow_pickle=False)
    freq_offset = stream_file.tell()
    np.save(stream_file, freqs, allow_pickle=False)
    time_offset = stream_file.tell()
    np.save(stream_file, times, allow_pickle=False)

    csv_writer.writerow(
        [
            record_index,
            wav_path.name,
            label,
            low,
            high,
            sample_rate,
            spectrogram.shape[0],
            spectrogram.shape[1],
            spec_offset,
            freq_offset,
            time_offset,
        ]
    )


def main() -> None:
    wav_files = sorted(WAV_DIR.glob("*.wav"))

    with ARRAY_OUTPUT_PATH.open("wb") as stream_file, INDEX_OUTPUT_PATH.open("w", newline="") as index_file:
        writer = csv.writer(index_file)
        writer.writerow(
            [
                "record_index",
                "wav_filename",
                "label",
                "low_frequency",
                "high_frequency",
                "sample_rate",
                "n_frequency_bins",
                "n_time_bins",
                "spectrogram_offset",
                "frequencies_offset",
                "times_offset",
            ]
        )

        for record_index, wav_path in enumerate(wav_files):
            append_event_arrays(stream_file, writer, record_index, wav_path)


if __name__ == "__main__":
    main()

# Summary:
# - Iterates over every WAV in ./waves.
# - Parses filename as <label>_<low_freq>_<high_freq>_<index>.wav.
# - Builds one spectrogram per WAV using nfft=512, winsize=256, overlap=98, scale=default.
# - Crops each spectrogram to its filename low/high frequency band.
# - Appends spectrogram/frequency/time arrays into one NumPy binary stream: ./spectrograms_arrays.npy
# - Writes an index CSV (offsets + metadata) to ./spectrograms_arrays_index.csv for retrieval.


In [3]:
from pathlib import Path
import csv
import numpy as np

ARRAY_PATH = Path("./spectrograms_arrays.npy")
INDEX_PATH = Path("./spectrograms_arrays_index.csv")
SAMPLE_ROW = 0  # change this to inspect another sample

if not ARRAY_PATH.exists() or not INDEX_PATH.exists():
    raise FileNotFoundError("Expected ./spectrograms_arrays.npy and ./spectrograms_arrays_index.csv to exist.")

with INDEX_PATH.open(newline="") as index_file:
    rows = list(csv.DictReader(index_file))

if not rows:
    raise ValueError("Index file is empty.")

sample_row = min(SAMPLE_ROW, len(rows) - 1)
row = rows[sample_row]
spectrogram_offset = int(row["spectrogram_offset"])

with ARRAY_PATH.open("rb") as stream_file:
    stream_file.seek(spectrogram_offset)
    sample_spectrogram = np.load(stream_file, allow_pickle=False)

print("Sample row:", sample_row)
print("Source wav:", row["wav_filename"])
print("Label:", row["label"])
print("Shape:", sample_spectrogram.shape)
print("Spectrogram array:")
print(sample_spectrogram)



Sample row: 0
Source wav: bma_11.4_30.3_49.wav
Label: bma
Shape: (39, 63)
Spectrogram array:
[[-57.547188 -56.89762  -56.291218 ... -45.289753 -45.059395 -44.894997]
 [-55.976505 -55.594902 -55.20992  ... -45.42886  -45.317142 -45.23528 ]
 [-57.076645 -57.513245 -57.93456  ... -47.30537  -47.495323 -47.689644]
 ...
 [-57.003113 -57.18388  -57.453114 ... -64.912766 -66.022255 -67.17082 ]
 [-55.532757 -55.516773 -55.56108  ... -64.51756  -66.37245  -67.97684 ]
 [-55.40602  -55.01828  -54.69065  ... -59.455837 -59.342857 -59.20357 ]]


# Resize And Normalize Spectrogram Arrays
Use this block after `spectrograms_arrays.npy` and `spectrograms_arrays_index.csv` are generated.
It resizes every spectrogram to a common shape, then applies dataset-level z-score normalization.


In [ ]:
import csv
from pathlib import Path

import numpy as np
from numpy.lib.format import open_memmap

ARRAY_PATH = Path("./spectrograms_arrays.npy")
INDEX_PATH = Path("./spectrograms_arrays_index.csv")
PREPROC_DIR = Path("./preprocessed")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

RESIZED_PATH = PREPROC_DIR / "X_resized.npy"
NORMALIZED_PATH = PREPROC_DIR / "X_normalized.npy"
LABELS_PATH = PREPROC_DIR / "y_labels.npy"
LABEL_NAMES_PATH = PREPROC_DIR / "label_names.npy"
STATS_PATH = PREPROC_DIR / "normalization_stats.npz"

FREQ_PERCENTILE = 90
TIME_PERCENTILE = 90
MAX_FREQ_BINS = 128
MAX_TIME_BINS = 256
ROUND_FREQ_TO = 8
ROUND_TIME_TO = 16

if not ARRAY_PATH.exists() or not INDEX_PATH.exists():
    raise FileNotFoundError("Expected ./spectrograms_arrays.npy and ./spectrograms_arrays_index.csv to exist.")

with INDEX_PATH.open(newline="") as index_file:
    rows = list(csv.DictReader(index_file))

if not rows:
    raise ValueError("Index file is empty.")


def _round_up(value: int, base: int) -> int:
    return int(np.ceil(value / base) * base)


freq_bins = np.array([int(r["n_frequency_bins"]) for r in rows], dtype=np.int32)
time_bins = np.array([int(r["n_time_bins"]) for r in rows], dtype=np.int32)

p_freq = int(np.percentile(freq_bins, FREQ_PERCENTILE))
p_time = int(np.percentile(time_bins, TIME_PERCENTILE))

target_freq_bins = min(MAX_FREQ_BINS, _round_up(max(1, p_freq), ROUND_FREQ_TO))
target_time_bins = min(MAX_TIME_BINS, _round_up(max(1, p_time), ROUND_TIME_TO))

print(f"Samples: {len(rows)}")
print(f"Frequency bins percentiles: p90={int(np.percentile(freq_bins, 90))}, p95={int(np.percentile(freq_bins, 95))}, max={freq_bins.max()}")
print(f"Time bins percentiles: p90={int(np.percentile(time_bins, 90))}, p95={int(np.percentile(time_bins, 95))}, max={time_bins.max()}")
print(f"Target shape selected: ({target_freq_bins}, {target_time_bins})")
print("If needed, change MAX_* or percentile constants above and rerun this section.")



Samples: 58570
Frequency bins percentiles: p90=96, p95=154, max=205
Time bins percentiles: p90=390, p95=456, max=1353
Target shape selected: (96, 256)
If needed, change MAX_* or percentile constants above and rerun this section.


In [5]:
def resize_2d_bilinear(x: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    """Resize 2D array with bilinear interpolation using only NumPy."""
    in_h, in_w = x.shape
    if in_h == out_h and in_w == out_w:
        return x.astype(np.float32, copy=True)

    x = x.astype(np.float32, copy=False)

    old_x = np.linspace(0.0, 1.0, in_w, dtype=np.float32)
    new_x = np.linspace(0.0, 1.0, out_w, dtype=np.float32)
    tmp = np.empty((in_h, out_w), dtype=np.float32)
    for i in range(in_h):
        tmp[i, :] = np.interp(new_x, old_x, x[i, :])

    old_y = np.linspace(0.0, 1.0, in_h, dtype=np.float32)
    new_y = np.linspace(0.0, 1.0, out_h, dtype=np.float32)
    out = np.empty((out_h, out_w), dtype=np.float32)
    for j in range(out_w):
        out[:, j] = np.interp(new_y, old_y, tmp[:, j])

    return out


def load_spectrogram(stream_file, spectrogram_offset: int) -> np.ndarray:
    stream_file.seek(spectrogram_offset)
    return np.load(stream_file, allow_pickle=False).astype(np.float32, copy=False)


num_samples = len(rows)
X_resized = open_memmap(
    RESIZED_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(num_samples, target_freq_bins, target_time_bins),
)

label_names = sorted({r["label"] for r in rows})
label_to_id = {name: idx for idx, name in enumerate(label_names)}
y = np.empty(num_samples, dtype=np.int32)

with ARRAY_PATH.open("rb") as stream_file:
    for i, row in enumerate(rows):
        spec = load_spectrogram(stream_file, int(row["spectrogram_offset"]))
        X_resized[i] = resize_2d_bilinear(spec, target_freq_bins, target_time_bins)
        y[i] = label_to_id[row["label"]]

        if (i + 1) % 2000 == 0:
            print(f"Resized {i + 1}/{num_samples}")

X_resized.flush()
np.save(LABELS_PATH, y)
np.save(LABEL_NAMES_PATH, np.array(label_names, dtype="U16"))

print("Resizing finished.")
print(f"Saved resized arrays to: {RESIZED_PATH}")
print(f"Saved labels to: {LABELS_PATH}")
print(f"Classes: {label_names}")



Resized 2000/58570
Resized 4000/58570
Resized 6000/58570
Resized 8000/58570
Resized 10000/58570
Resized 12000/58570
Resized 14000/58570
Resized 16000/58570
Resized 18000/58570
Resized 20000/58570
Resized 22000/58570
Resized 24000/58570
Resized 26000/58570
Resized 28000/58570
Resized 30000/58570
Resized 32000/58570
Resized 34000/58570
Resized 36000/58570
Resized 38000/58570
Resized 40000/58570
Resized 42000/58570
Resized 44000/58570
Resized 46000/58570
Resized 48000/58570
Resized 50000/58570
Resized 52000/58570
Resized 54000/58570
Resized 56000/58570
Resized 58000/58570
Resizing finished.
Saved resized arrays to: preprocessed/X_resized.npy
Saved labels to: preprocessed/y_labels.npy
Classes: ['bma', 'bmb', 'bmd', 'bmz', 'bp20', 'bp20plus', 'bpd']


In [6]:
# Dataset-level z-score normalization (training-only scenario)
X_resized = np.load(RESIZED_PATH, mmap_mode="r")
num_samples = X_resized.shape[0]

sum_x = np.float64(0.0)
sum_x2 = np.float64(0.0)
count = np.int64(0)

for i in range(num_samples):
    arr = X_resized[i]
    sum_x += arr.sum(dtype=np.float64)
    sum_x2 += np.square(arr, dtype=np.float64).sum(dtype=np.float64)
    count += arr.size

mean = sum_x / count
var = (sum_x2 / count) - (mean ** 2)
std = float(np.sqrt(max(var, 1e-12)))

X_norm = open_memmap(
    NORMALIZED_PATH,
    mode="w+",
    dtype=np.float32,
    shape=X_resized.shape,
)

for i in range(num_samples):
    X_norm[i] = (X_resized[i] - mean) / std
    if (i + 1) % 2000 == 0:
        print(f"Normalized {i + 1}/{num_samples}")

X_norm.flush()
np.savez(
    STATS_PATH,
    mean=np.float32(mean),
    std=np.float32(std),
    target_freq_bins=np.int32(target_freq_bins),
    target_time_bins=np.int32(target_time_bins),
    num_samples=np.int32(num_samples),
)

print("Normalization finished.")
print(f"Saved normalized arrays to: {NORMALIZED_PATH}")
print(f"Saved stats to: {STATS_PATH}")
print(f"mean={mean:.6f}, std={std:.6f}")



Normalized 2000/58570
Normalized 4000/58570
Normalized 6000/58570
Normalized 8000/58570
Normalized 10000/58570
Normalized 12000/58570
Normalized 14000/58570
Normalized 16000/58570
Normalized 18000/58570
Normalized 20000/58570
Normalized 22000/58570
Normalized 24000/58570
Normalized 26000/58570
Normalized 28000/58570
Normalized 30000/58570
Normalized 32000/58570
Normalized 34000/58570
Normalized 36000/58570
Normalized 38000/58570
Normalized 40000/58570
Normalized 42000/58570
Normalized 44000/58570
Normalized 46000/58570
Normalized 48000/58570
Normalized 50000/58570
Normalized 52000/58570
Normalized 54000/58570
Normalized 56000/58570
Normalized 58000/58570
Normalization finished.
Saved normalized arrays to: preprocessed/X_normalized.npy
Saved stats to: preprocessed/normalization_stats.npz
mean=-58.636062, std=12.152215


In [7]:
# Quick sanity check sample
Xn = np.load(NORMALIZED_PATH, mmap_mode="r")
y = np.load(LABELS_PATH)
label_names = np.load(LABEL_NAMES_PATH)
stats = np.load(STATS_PATH)

sample_idx = 0
sample = Xn[sample_idx]
label_id = y[sample_idx]

print("Final tensor shape:", Xn.shape)
print("Sample shape:", sample.shape)
print("Sample label:", label_names[label_id])
print("Sample stats (min/mean/max/std):", float(sample.min()), float(sample.mean()), float(sample.max()), float(sample.std()))
print("Global mean/std used:", float(stats["mean"]), float(stats["std"]))
print("Sample array:")
print(sample)



Final tensor shape: (58570, 96, 256)
Sample shape: (96, 256)
Sample label: bma
Sample stats (min/mean/max/std): -2.496019124984741 0.3669091761112213 1.62887442111969 0.5684331655502319
Global mean/std used: -58.63606262207031 12.152215003967285
Sample array:
[[ 0.08960291  0.10259908  0.11559556 ...  1.1241671   1.1274565
   1.1307456 ]
 [ 0.1413032   0.15215474  0.16300659 ...  1.1142877   1.1169164
   1.1195451 ]
 [ 0.19300348  0.2017107   0.21041793 ...  1.104408    1.1063763
   1.1083444 ]
 ...
 [ 0.2574554   0.2592626   0.26107007 ... -0.5740133  -0.5991361
  -0.62425894]
 [ 0.26162696  0.26640937  0.27119207 ... -0.3131428  -0.32431078
  -0.33547872]
 [ 0.26579848  0.27355614  0.2813141  ... -0.05227356 -0.04948699
  -0.0467001 ]]
